# Model Training with Citation Graph

This notebook extends the TF-IDF baseline with **citation-graph features** built from OpenAlex reference metadata.

- **Graph**: directed edges `u → v` when paper `u` cites paper `v` (both in train+test corpus).
- **Features**: in/out degree, PageRank, and neighbor label distribution (train labels only).
- **Model**: TF-IDF on title+abstract + graph features → logistic regression.

First run builds `data/citation_graph_cache.pkl` (OpenAlex API, ~1–3 min).

## 1. Imports and data paths

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler

from citation_graph import (
    build_citation_graph,
    build_feature_matrix,
    graph_summary,
)

ROOT = Path(".")
TRAIN_PATH = ROOT / "CSV" / "train_cleaned_phong.csv"
TEST_PATHS = [
    ROOT / "CSV" / "public_test_with_abstracts_updated_4_2.csv",
    ROOT / "CSV" / "private_test_with_abstracts_update.csv",
]

## 2. Load and clean data

In [2]:
data = pd.read_csv(TRAIN_PATH)
data_test = pd.concat([pd.read_csv(p) for p in TEST_PATHS], ignore_index=True)

for df in (data, data_test):
    if "authors" in df.columns:
        df["authors"] = df["authors"].fillna("unknown")

# Training: require non-empty abstracts (same as model_training.ipynb)
data_filtered = data[
    data["abstract"].notna() & (data["abstract"].astype(str).str.strip() != "")
].copy()

print(f"Train: {len(data_filtered)} rows (dropped {len(data) - len(data_filtered)} without abstract)")
print(f"Test:  {len(data_test)} rows")
print(data_filtered["Label"].value_counts().sort_index())

Train: 1933 rows (dropped 0 without abstract)
Test:  596 rows
Label
1    696
2    412
3    342
4    271
5    212
Name: count, dtype: int64


## 3. Build citation graph (OpenAlex)

In [3]:
corpus = pd.concat([data_filtered, data_test], ignore_index=True)
G = build_citation_graph(corpus)
print(graph_summary(G))

{'nodes': 2529, 'edges': 691, 'density': 0.00010808169537166339, 'weakly_connected_components': 1959}


## 4. Graph + text features

In [4]:
train_ids = set(data_filtered["id"].astype(int))
test_ids = set(data_test["id"].astype(int))
all_ids = list(data_filtered["id"].astype(int)) + list(data_test["id"].astype(int))

labels_by_id = dict(zip(data_filtered["id"].astype(int), data_filtered["Label"].astype(int)))

X_graph_all = build_feature_matrix(G, labels_by_id, train_ids, all_ids)
n_train = len(data_filtered)
X_graph_train = X_graph_all[:n_train]
X_graph_test = X_graph_all[n_train:]

scaler = StandardScaler()
X_graph_train_scaled = scaler.fit_transform(X_graph_train)
X_graph_test_scaled = scaler.transform(X_graph_test)

text_train = (
    data_filtered["title"].astype(str) + " " + data_filtered["abstract"].astype(str)
)
text_test = data_test["title"].astype(str) + " " + data_test["abstract"].fillna("").astype(str)

tfidf = TfidfVectorizer(max_features=5000)
X_text_train = tfidf.fit_transform(text_train)
X_text_test = tfidf.transform(text_test)

X_train = hstack([X_text_train, csr_matrix(X_graph_train_scaled)])
X_test = hstack([X_text_test, csr_matrix(X_graph_test_scaled)])
y_train = data_filtered["Label"].astype(int)

print(f"Feature matrix: text {X_text_train.shape[1]} + graph {X_graph_train.shape[1]} = {X_train.shape[1]} dims")

Feature matrix: text 5000 + graph 8 = 5008 dims


## 5. Cross-validation (train only)

In [5]:
model_cv = LogisticRegression(max_iter=2000, random_state=42, class_weight="balanced")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(model_cv, X_train, y_train, cv=cv)

print("Citation graph + TF-IDF (5-fold CV)")
print(classification_report(y_train, y_pred_cv))
print(f"Macro F1: {f1_score(y_train, y_pred_cv, average='macro'):.4f}")

Citation graph + TF-IDF (5-fold CV)
              precision    recall  f1-score   support

           1       0.54      0.51      0.53       696
           2       0.27      0.25      0.26       412
           3       0.23      0.21      0.22       342
           4       0.24      0.25      0.24       271
           5       0.36      0.51      0.42       212

    accuracy                           0.36      1933
   macro avg       0.33      0.34      0.33      1933
weighted avg       0.37      0.36      0.36      1933

Macro F1: 0.3325


## 6. Baseline comparison (text only)

In [6]:
model_text = LogisticRegression(max_iter=2000, random_state=42, class_weight="balanced")
y_pred_text_cv = cross_val_predict(model_text, X_text_train, y_train, cv=cv)

print("TF-IDF only (5-fold CV)")
print(classification_report(y_train, y_pred_text_cv))
print(f"Macro F1: {f1_score(y_train, y_pred_text_cv, average='macro'):.4f}")

TF-IDF only (5-fold CV)
              precision    recall  f1-score   support

           1       0.52      0.53      0.52       696
           2       0.26      0.24      0.25       412
           3       0.25      0.20      0.22       342
           4       0.24      0.25      0.25       271
           5       0.37      0.51      0.43       212

    accuracy                           0.37      1933
   macro avg       0.33      0.35      0.34      1933
weighted avg       0.36      0.37      0.36      1933

Macro F1: 0.3351


## 7. Train final model and predict test set

In [7]:
model_final = LogisticRegression(max_iter=2000, random_state=42, class_weight="balanced")
model_final.fit(X_train, y_train)
predictions = model_final.predict(X_test).astype(int)

results_df = pd.DataFrame({"id": data_test["id"], "Label": predictions})
results_df.to_csv("predictions_citation_graph.csv", index=False)

print(f"Saved predictions_citation_graph.csv ({len(results_df)} rows)")
print(results_df.head(10))

Saved predictions_citation_graph.csv (596 rows)
     id  Label
0   979      5
1  1106      3
2  1894      2
3  1718      1
4  1989      3
5  2010      1
6  2356      1
7  1179      1
8  3001      2
9   796      5


## 8. Optional: label propagation on the graph

Simple harmonic-style propagation using only train labels (for comparison).

In [8]:
def label_propagation(G, labels_by_id, train_ids, node_ids, n_classes=5, alpha=0.6, max_iter=30):
    n = len(node_ids)
    idx = {nid: i for i, nid in enumerate(node_ids)}
    Y = np.zeros((n, n_classes), dtype=np.float64)
    known = np.zeros(n, dtype=bool)
    for nid in node_ids:
        if nid in train_ids and nid in labels_by_id:
            Y[idx[nid], int(labels_by_id[nid]) - 1] = 1.0
            known[idx[nid]] = True
    global_prior = np.bincount(
        [int(labels_by_id[i]) - 1 for i in train_ids if i in labels_by_id],
        minlength=n_classes,
    ).astype(float)
    global_prior /= global_prior.sum() or 1.0

    A = np.zeros((n, n), dtype=np.float64)
    for nid in node_ids:
        if nid not in G:
            continue
        i = idx[nid]
        nbrs = list(G.predecessors(nid)) + list(G.successors(nid))
        for nb in nbrs:
            if nb in idx:
                A[i, idx[nb]] = 1.0
    row_sum = A.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    A_norm = A / row_sum

    F = Y.copy()
    for _ in range(max_iter):
        F_new = alpha * (A_norm @ F) + (1 - alpha) * Y
        F_new[~known] = F_new[~known]
        F_new[known] = Y[known]
        if np.allclose(F_new, F, atol=1e-5):
            break
        F = F_new
    isolated = A.sum(axis=1) == 0
    F[isolated] = global_prior
    return F.argmax(axis=1) + 1

lp_preds = label_propagation(G, labels_by_id, train_ids, list(data_test["id"].astype(int)))
pd.DataFrame({"id": data_test["id"], "Label": lp_preds}).to_csv(
    "predictions_citation_graph_label_propagation.csv", index=False
)
print("Saved predictions_citation_graph_label_propagation.csv")

Saved predictions_citation_graph_label_propagation.csv
